<a href="https://colab.research.google.com/github/TBDA-Classes/2026G3/blob/main/P7_and_F5_2_data_quality_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Read the Excel file: all CSV-like rows are currently stored in column A
raw = pd.read_excel(
    "/content/cedint_f1_hourly_clean.xlsx",
    header=None
)

print("Raw shape:", raw.shape)
print(raw.head())

Raw shape: (470681, 1)
                                                   0
0  timestamp_utc,timestamp_local,zone,zone_code,l...
1  2020-04-02 23:00:00,2020-04-03 01:00:00,NE,1NE...
2  2020-04-03 00:00:00,2020-04-03 02:00:00,NE,1NE...
3  2020-04-03 01:00:00,2020-04-03 03:00:00,NE,1NE...
4  2020-04-03 02:00:00,2020-04-03 04:00:00,NE,1NE...


In [2]:
from io import StringIO

# Join all rows back into CSV text
csv_text = "\n".join(raw.iloc[:, 0].astype(str))

# Parse correctly using comma as separator
df = pd.read_csv(
    StringIO(csv_text),
    low_memory=False
)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
display(df.head())

Dataset shape: (470680, 55)

Columns:
['timestamp_utc', 'timestamp_local', 'zone', 'zone_code', 'load_type', 'n_devices', 'era', 'recommendation', 'usable_for_model', 'energy_wh_model', 'energy_source', 'energy_wh', 'energy_method', 'power_w', 'apparent_power_va', 'current_a', 'power_w_raw', 'counter_wh', 'counter_wh_raw', 'segment', 'temp_c', 'hum_pct', 'lum_lux', 'occupancy', 'source_rows', 'complete_rows', 'flag_missing_hour', 'flag_zero_record', 'flag_partial_sum', 'flag_counter_spike', 'flag_counter_reset', 'flag_energy_implausible', 'flag_neg_noise', 'flag_neg_large', 'reversed_sign', 'flag_low_coverage', 'flag_power_spike', 'flag_power_energy_mismatch', 'flag_env_missing', 'flag_temp_impossible', 'flag_temp_extreme', 'flag_hum_invalid', 'flag_lum_day_zero', 'series_excluded', 'is_holiday', 'holiday_name', 'is_weekend', 'is_august_low_use', 'flag_covid_2020', 'n_zero_record', 'n_partial', 'n_reset', 'n_spike', 'n_neg_noise', 'n_neg_large']

First rows:


,timestamp_utc,timestamp_local,zone,zone_code,load_type,n_devices,era,recommendation,usable_for_model,energy_wh_model,...,holiday_name,is_weekend,is_august_low_use,flag_covid_2020,n_zero_record,n_partial,n_reset,n_spike,n_neg_noise,n_neg_large
0,2020-04-02 23:00:00,2020-04-03 01:00:00,NE,1NE,HVAC,2,hourly,Use,1,3.841,...,NaN,0,0,1,0,0,0,0,0,0
1,2020-04-03 00:00:00,2020-04-03 02:00:00,NE,1NE,HVAC,2,hourly,Use,1,3.763,...,NaN,0,0,1,0,0,0,0,0,0
2,2020-04-03 01:00:00,2020-04-03 03:00:00,NE,1NE,HVAC,2,hourly,Use,1,3.683,...,NaN,0,0,1,0,0,0,0,0,0
3,2020-04-03 02:00:00,2020-04-03 04:00:00,NE,1NE,HVAC,2,hourly,Use,1,3.720,...,NaN,0,0,1,0,0,0,0,0,0
4,2020-04-03 03:00:00,2020-04-03 05:00:00,NE,1NE,HVAC,2,hourly,Use,1,3.775,...,NaN,0,0,1,0,0,0,0,0,0


In [3]:
print("Shape:", df.shape)

print("\nZones:")
print(df["zone"].value_counts())

print("\nLoad types:")
print(df["load_type"].value_counts())

print("\nUsable for model:")
print(df["usable_for_model"].value_counts(dropna=False))

print("\nNegative power values:")
print((df["power_w"] < 0).sum())

print("\nNegative power values by zone/load:")
display(
    df[df["power_w"] < 0]
    .groupby(["zone", "load_type"])
    .size()
    .reset_index(name="negative_rows")
)

Shape: (470680, 55)

Zones:
zone
NW    160980
SE    134111
SW    121929
NE     53660
Name: count, dtype: int64

Load types:
load_type
Plugs & Sockets    107320
Lighting            80490
HVAC                80477
Emerg. Lighting     80464
Hand Dryer          53660
RACK                41452
Projector           26817
Name: count, dtype: int64

Usable for model:
usable_for_model
1    437158
0     33522
Name: count, dtype: int64

Negative power values:
0

Negative power values by zone/load:


,zone,load_type,negative_rows


In [4]:
# Keep only valid rows for the updated NW Emergency Lighting series
nw_emerg = df[
    (df["zone"] == "NW") &
    (df["load_type"] == "Emerg. Lighting") &
    (df["usable_for_model"] == 1)
].copy()

print("Rows:", len(nw_emerg))

print("\nPower summary:")
print(nw_emerg["power_w"].describe())

print("\nNegative power values:")
print((nw_emerg["power_w"] < 0).sum())

print("\nMissing environmental values:")
print(
    nw_emerg[["temp_c", "hum_pct", "lum_lux", "occupancy"]]
    .isna()
    .sum()
)

Rows: 26830

Power summary:
count    24636.000000
mean        12.443348
std          4.382220
min          0.000000
25%         11.265000
50%         13.707000
75%         15.326925
max         21.304000
Name: power_w, dtype: float64

Negative power values:
0

Missing environmental values:
temp_c       2211
hum_pct      2211
lum_lux      2211
occupancy    2211
dtype: int64


In [5]:
from scipy.stats import pearsonr, spearmanr
import pandas as pd

drivers = ["temp_c", "hum_pct", "lum_lux", "occupancy"]

results = []

for driver in drivers:
    temp = nw_emerg[[driver, "power_w"]].dropna()

    pearson = pearsonr(temp[driver], temp["power_w"])[0]
    spearman = spearmanr(temp[driver], temp["power_w"])[0]

    results.append({
        "driver": driver,
        "n": len(temp),
        "pearson": pearson,
        "spearman": spearman
    })

corr_new = pd.DataFrame(results)

display(corr_new)

,driver,n,pearson,spearman
0,temp_c,24488,-0.230104,-0.471774
1,hum_pct,24488,0.050418,0.067763
2,lum_lux,24488,-0.049228,-0.081626
3,occupancy,24488,0.133637,0.196438


In [6]:
import statsmodels.api as sm
import pandas as pd

# Prepare hour variable
nw_emerg["timestamp_local"] = pd.to_datetime(nw_emerg["timestamp_local"])
nw_emerg["hour"] = nw_emerg["timestamp_local"].dt.hour

drivers = ["temp_c", "hum_pct", "lum_lux", "occupancy"]

partial_results = []

for driver in drivers:

    temp = nw_emerg[[driver, "power_w", "hour"]].dropna().copy()

    # Hour as categorical variable
    hour_dummies = pd.get_dummies(
        temp["hour"],
        prefix="hour",
        drop_first=True,
        dtype=float
    )

    X = sm.add_constant(hour_dummies)

    # Remove the effect of hour from the environmental variable
    model_driver = sm.OLS(temp[driver].astype(float), X).fit()
    resid_driver = model_driver.resid

    # Remove the effect of hour from power
    model_power = sm.OLS(temp["power_w"].astype(float), X).fit()
    resid_power = model_power.resid

    partial_corr = resid_driver.corr(resid_power)

    partial_results.append({
        "driver": driver,
        "n": len(temp),
        "partial_corr_hour": partial_corr
    })

partial_new = pd.DataFrame(partial_results)

display(partial_new)

,driver,n,partial_corr_hour
0,temp_c,24488,-0.225698
1,hum_pct,24488,0.050193
2,lum_lux,24488,-0.077946
3,occupancy,24488,0.134765



After the data-cleaning update, the NW Emergency Lighting series was rechecked because it had previously contained negative power values caused by a reversed current clamp.

In the corrected dataset, no negative power values remain. Pearson, Spearman and partial correlations with indoor temperature, humidity, luminosity and occupancy were recalculated.

The results remained practically unchanged. In particular, the partial correlation between indoor temperature and Emergency Lighting remained approximately -0.226.

Therefore, the main conclusions of the F5.2 environmental-driver analysis remain valid after the data-quality correction.